In [1]:
import pandas as pd

# File paths
input_file = 'test_tag_columns.csv'
output_file = 'transformed_text_dosage.csv'

# 1. Read the CSV file directly from the workplace
df = pd.read_csv(input_file)

# 2. Select only 'text' and 'Dosage' columns
filtered_df = df[['text', 'Dosage']]

# 3. Save the filtered dataset to a new CSV file
filtered_df.to_csv(output_file, index=False, encoding='utf-8-sig')

# 4. Preview the results
print("--- Data Transformed Successfully ---")
print(filtered_df.head())
print(f"\nSaved to: {output_file}")
print(f"Total Rows: {filtered_df.shape[0]} | Total Columns: {filtered_df.shape[1]}")

--- Data Transformed Successfully ---
                                                text Dosage
0  আমি < NAME > । আমার বয়স 27 বছর । সাম্প্রতিক স...    NaN
1  আপনার প্রশ্নের জন্য ধন্যবাদ । দুশ্চিন্তা কমান ...    NaN
2  Thank you for your question . Your serum Trigl...    NaN
3  Proshno korar jonno dhonnobad . Ei boyosher ba...    NaN
4  বেশ কয়েকদিন যাবত গলায় খুব ব্যাথা সেই সাথে কাশি...    NaN

Saved to: transformed_text_dosage.csv
Total Rows: 3179 | Total Columns: 2


In [2]:
!pip install -q openai tqdm

In [3]:
import time
import pandas as pd
from tqdm import tqdm
from google.colab import userdata
from openai import OpenAI

# 1. Fetch API key from Colab Secrets
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    raise ValueError("Key 'OPENROUTER_API_KEY' not found in Colab Secrets. Please check the Secrets tab.") from e

# 2. Initialize OpenAI client configured for OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# Set model name
MODEL_NAME = "gpt-5.6-luna"

# 3. Prompt Template adapted for Dosage
PROMPT_TEMPLATE = """You are given a Bangla medical sentence containing one or more dosage entities.

Your task is to create a modified version of the sentence by replacing exactly ONE dosage entity with a different but medically plausible dosage.

Rules:
1. Identify the specified dosage entity in the sentence.
2. Replace exactly ONE occurrence of that dosage with another dosage.
3. The replacement must be a different dosage from the original dosage.
4. The replacement must NOT be another dosage already present in the original sentence.
5. The replacement should be plausible in the same medical context.
6. Maintain the realistic form/unit (e.g., mg, ml, tablet count) appropriate to the context.
7. The replacement should fit naturally into the surrounding sentence without making the sentence medically or linguistically implausible.
8. Do not add any additional dosage.
9. Do not remove, add, or modify any other information in the sentence.
10. Keep the ENTIRE sentence structure intact. Do NOT truncate, cut short, or summarize any part of the original text.
11. Do not modify the original NER annotation.
12. Output ONLY the complete modified Bangla sentence from start to finish.
13. Do not provide explanations or identify the replacement.

Example:

Original sentence:
রোগীকে দিনে ২ বার ৫০০ মিলিগ্রাম ঔষধ খেতে বলা হয়েছে।

Dosage entity:
৫০০ মিলিগ্রাম

Output:
রোগীকে দিনে ২ বার ২৫০ মিলিগ্রাম ঔষধ খেতে বলা হয়েছে।

Now perform the replacement.

Original sentence:
{SENTENCE}

Dosage entity:
{DOSAGE_ENTITY}

Modified sentence:"""

# 4. Load the filtered dataset
input_file = "transformed_text_dosage.csv"
output_file = "transformed_with_luna_modified_dosage.csv"

df = pd.read_csv(input_file)

# 5. Helper function to process individual rows
def get_modified_sentence(sentence, dosage_entity):
    if pd.isna(dosage_entity) or str(dosage_entity).strip() == "":
        return None  # Skip if no dosage entity is present

    prompt = PROMPT_TEMPLATE.format(
        SENTENCE=str(sentence).strip(),
        DOSAGE_ENTITY=str(dosage_entity).strip()
    )

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"\nError processing entity '{dosage_entity}': {e}")
        return None

# 6. Iterate through DataFrame and perform replacement
modified_sentences = []

print(f"Processing {len(df)} rows using {MODEL_NAME} via OpenRouter...")

for idx, row in tqdm(df.iterrows(), total=len(df)):
    sentence = row['text']
    dosage = row['Dosage']

    modified_text = get_modified_sentence(sentence, dosage)
    modified_sentences.append(modified_text)

    # Optional small delay to respect API rate limits
    time.sleep(0.1)

# 7. Add results to DataFrame and export
df['modified_text'] = modified_sentences

# Save output
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\nProcessing complete! File saved as '{output_file}'.")

Processing 3179 rows using gpt-5.6-luna via OpenRouter...


100%|██████████| 3179/3179 [27:09<00:00,  1.95it/s]


Processing complete! File saved as 'transformed_with_luna_modified_dosage.csv'.


In [4]:
import pandas as pd

# Load the output file from your current run
df_final = pd.read_csv("transformed_with_luna_modified_dosage.csv")

# Drop rows where 'Dosage' or 'modified_text' is missing
df_clean = df_final.dropna(subset=['Dosage', 'modified_text']).copy()

# Save as clean dataset
df_clean.to_csv("transformed_with_luna_modified_dosage_cleaned.csv", index=False, encoding='utf-8-sig')

print(f"Original rows: {len(df_final)} | Cleaned rows: {len(df_clean)}")

Original rows: 3179 | Cleaned rows: 495
